# Course 1, Week 4 — Core Neural Network Components

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #4](https://github.com/majorgilles/pytorch_for_deep_learning/issues/4)

**Focus:** Compose layers, activations, losses, and convolutional components into networks.

## Video guide

| Video | Covered notebook section |
|---|---|
| Introduction to convolutional neural networks | From pixels to learned spatial features |

> Open the [DeepLearning.AI course lesson list](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson) to watch the course video.


## From pixels to learned spatial features

A nature classifier may need to distinguish flowers, butterflies, and small animals using local visual features such as edges, wing veins, antennae, spots, and textures. A fully connected layer can combine pixels, but after flattening an image it has no built-in notion that nearby pixels are spatial neighbors. It also learns separate weights for every input position.

A **convolutional neural network (CNN)** adds two useful assumptions:

1. **Local connectivity:** each output examines a small neighborhood rather than the entire image.
2. **Weight sharing:** the same filter scans every spatial position, allowing one learned pattern to be recognized anywhere in the image.

This resembles the biological observation that visual-cortex neurons respond to patterns within local receptive fields. Early CNN layers often respond to simple features such as edges and textures; deeper layers combine them into task-specific patterns.


### How a convolution scans an image

A kernel, or filter, is a small grid of weights such as $3 \times 3$. At each image position, the kernel weights are multiplied elementwise by the pixels beneath them and the products are summed:

$$
y_{i,j} = b + \sum_{c=1}^{C_{\text{in}}}\sum_{u=1}^{K_h}\sum_{v=1}^{K_w}
W_{c,u,v}\,x_{c,i+u,j+v}.
$$

Sliding the same kernel across the image creates one **feature map**. Different kernels produce different feature maps: one might respond strongly to vertical contrast, another to horizontal contrast, and others to textures or color patterns. In a trainable CNN, these weights are learned through backpropagation rather than designed by hand.

> PyTorch's `Conv2d` computes cross-correlation—the kernel is not flipped—but deep-learning literature conventionally calls the operation convolution.


In [1]:
import torch
import torch.nn.functional as F
from torch import nn

# Shape [N, C, H, W] = [1 image, 1 grayscale channel, 5 rows, 5 columns].
grayscale_image: torch.Tensor = torch.tensor(
    [[[[0, 0, 0, 1, 1],
       [0, 0, 0, 1, 1],
       [0, 0, 0, 1, 1],
       [0, 0, 0, 1, 1],
       [0, 0, 0, 1, 1]]]],
    dtype=torch.float32,
)

# Shape [C_out, C_in, K_h, K_w] = [1, 1, 3, 3].
vertical_edge_kernel: torch.Tensor = torch.tensor(
    [[[[-1, 0, 1],
       [-1, 0, 1],
       [-1, 0, 1]]]],
    dtype=torch.float32,
)

# Padding keeps the output's spatial dimensions at 5 x 5.
vertical_edges: torch.Tensor = F.conv2d(
    grayscale_image,
    vertical_edge_kernel,
    padding=1,
)

print(f"Input shape [N, C, H, W]: {grayscale_image.shape}")
print(f"Kernel shape [C_out, C_in, K_h, K_w]: {vertical_edge_kernel.shape}")
print(f"Edge-map shape [N, C_out, H_out, W_out]: {vertical_edges.shape}")
vertical_edges[0, 0]


Input shape [N, C, H, W]: torch.Size([1, 1, 5, 5])
Kernel shape [C_out, C_in, K_h, K_w]: torch.Size([1, 1, 3, 3])
Edge-map shape [N, C_out, H_out, W_out]: torch.Size([1, 1, 5, 5])


tensor([[ 0.,  0.,  2.,  2., -2.],
        [ 0.,  0.,  3.,  3., -3.],
        [ 0.,  0.,  3.,  3., -3.],
        [ 0.,  0.,  3.,  3., -3.],
        [ 0.,  0.,  2.,  2., -2.]])

### Configure `nn.Conv2d`

```python
nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1,
)
```

- **`in_channels`** is the number of input channels: usually $3$ for RGB and $1$ for grayscale.
- **`out_channels`** is the number of learned filters and therefore the number of output feature maps.
- **`kernel_size`** controls each filter's receptive field. A $3 \times 3$ kernel examines a pixel and its immediate neighbors.
- **`stride`** controls how far the kernel moves. Larger strides reduce spatial resolution and computation but may skip fine details.
- **`padding`** adds values, usually zeros, around the image boundary. With a $3 \times 3$ kernel, stride $1$, and padding $1$, edge pixels can be centered beneath the kernel and spatial size is preserved.

For dilation $d$, kernel size $K$, padding $P$, and stride $S$, one output dimension is

$$
H_{\text{out}} = \left\lfloor
\frac{H_{\text{in}} + 2P - d(K-1) - 1}{S} + 1
\right\rfloor.
$$


In [2]:
# A batch of four RGB images with shape [N, C_in, H_in, W_in].
rgb_batch: torch.Tensor = torch.rand(4, 3, 128, 128)

# Sixteen trainable filters each inspect all 3 input channels over a 3 x 3 area.
convolution: nn.Conv2d = nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1,
)
feature_maps: torch.Tensor = convolution(rgb_batch)

print(f"Input [N, C_in, H_in, W_in]: {rgb_batch.shape}")
print(f"Weights [C_out, C_in, K_h, K_w]: {convolution.weight.shape}")
print(f"Output [N, C_out, H_out, W_out]: {feature_maps.shape}")

assert feature_maps.shape == (4, 16, 128, 128)


Input [N, C_in, H_in, W_in]: torch.Size([4, 3, 128, 128])
Weights [C_out, C_in, K_h, K_w]: torch.Size([16, 3, 3, 3])
Output [N, C_out, H_out, W_out]: torch.Size([4, 16, 128, 128])


### Learning checkpoint

You should now be able to explain why CNNs preserve spatial structure, how a filter creates a feature map, and how `in_channels`, `out_channels`, `kernel_size`, `stride`, and `padding` affect `nn.Conv2d`.

The next section combines convolution, activation, pooling, flattening, and a fully connected layer into a complete classifier.


## Build a complete convolutional neural network

A complete CNN turns raw pixels into class scores in stages:

1. **Convolution** learns local filters and produces feature maps.
2. **ReLU** replaces negative activations with zero, introducing nonlinearity.
3. **Max pooling** compresses each feature map while retaining its strongest local responses.
4. Repeating convolution and pooling builds progressively richer, more compact features.
5. **Flattening** converts the feature maps into one vector per image.
6. A **fully connected** (`nn.Linear`) layer combines those features into class scores.

Like earlier models, a CNN subclasses `nn.Module`: layers are declared in `__init__()` and their data flow is defined in `forward()`.


### Compress feature maps with max pooling

`nn.MaxPool2d(kernel_size=2, stride=2)` divides each feature map into non-overlapping $2 \times 2$ regions and retains one maximum from each region. It halves both spatial dimensions, so the number of spatial values becomes one quarter:

$$
[H,W] \rightarrow \left[\frac{H}{2},\frac{W}{2}\right],
\qquad
H W \rightarrow \frac{H W}{4}.
$$

Pooling does not change the number of channels. It reduces computation in later layers and makes a strong feature response less sensitive to a small shift in position. It is deliberately lossy: weaker values are discarded.


In [3]:
# Shape [N, C, H, W] = [1 feature map, 1 channel, 4 rows, 4 columns].
feature_map: torch.Tensor = torch.tensor(
    [[[[1, 3, 2, 4],
       [5, 6, 7, 8],
       [2, 1, 9, 3],
       [4, 7, 5, 6]]]],
    dtype=torch.float32,
)

max_pool: nn.MaxPool2d = nn.MaxPool2d(kernel_size=2, stride=2)
pooled_map: torch.Tensor = max_pool(feature_map)

print(f"Before pooling [N, C, H, W]: {feature_map.shape}")
print(f"After pooling [N, C, H/2, W/2]: {pooled_map.shape}")
print(pooled_map[0, 0])

assert torch.equal(pooled_map, torch.tensor([[[[6.0, 8.0], [7.0, 9.0]]]]))


Before pooling [N, C, H, W]: torch.Size([1, 1, 4, 4])
After pooling [N, C, H/2, W/2]: torch.Size([1, 1, 2, 2])
tensor([[6., 8.],
        [7., 9.]])


### Follow the tensor shapes

For a batch of $N$ grayscale images resized to $28 \times 28$, two padded convolutions preserve spatial size and two pooling layers halve it twice:

| Stage | Tensor shape | Meaning |
|---|---:|---|
| Input | $[N, 1, 28, 28]$ | One grayscale channel |
| `Conv2d(1, 32, 3, padding=1)` | $[N, 32, 28, 28]$ | 32 learned feature maps |
| `ReLU` | $[N, 32, 28, 28]$ | Shape unchanged |
| `MaxPool2d(2)` | $[N, 32, 14, 14]$ | Width and height halved |
| `Conv2d(32, 64, 3, padding=1)` | $[N, 64, 14, 14]$ | 64 learned feature maps |
| `ReLU` | $[N, 64, 14, 14]$ | Shape unchanged |
| `MaxPool2d(2)` | $[N, 64, 7, 7]$ | Width and height halved again |
| `Flatten(start_dim=1)` | $[N, 64 \cdot 7 \cdot 7]$ | 3,136 features per image |
| `Linear(64*7*7, K)` | $[N, K]$ | One score for each of $K$ classes |

The first convolution contains 32 filters with weight shape $[32,1,3,3]$. Each filter has nine kernel weights for its one input channel, plus a learned bias by default. The second layer receives all 32 first-layer maps, so each of its 64 filters has shape $[32,3,3]$.


In [4]:
class NatureCNN(nn.Module):
    """Classify batches of 28 x 28 grayscale images.

    Args:
        num_classes: Number of output classes, ``K``.

    Shape:
        Input: ``[N, 1, 28, 28]``.
        Output: ``[N, K]`` unnormalized class scores (logits).
    """

    def __init__(self, num_classes: int) -> None:
        super().__init__()
        # First convolutional layer
        self.conv1: nn.Conv2d = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1: nn.MaxPool2d = nn.MaxPool2d(kernel_size=2)

        # Second convolutional layer
        self.conv2: nn.Conv2d = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2: nn.MaxPool2d = nn.MaxPool2d(kernel_size=2)

        # Flatten layer (no parameters, just reshaping)
        self.flatten = nn.Flatten()

        # Fully connected layer
        self.classifier: nn.Linear = nn.Linear(64 * 7 * 7, num_classes)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """Map images of shape ``[N, 1, 28, 28]`` to logits ``[N, K]``."""
        features: torch.Tensor = self.pool1(self.relu1(self.conv1(images)))
        # [N, 32, 28, 28] -> [N, 32, 14, 14]

        features = self.pool2(self.relu2(self.conv2(features)))
        # [N, 64, 14, 14] -> [N, 64, 7, 7]

        flattened: torch.Tensor = self.flatten(features)
        # Preserve N and flatten [64, 7, 7] into 3,136 features per image.
        return self.classifier(flattened)


In [6]:
num_classes: int = 10
model: NatureCNN = NatureCNN(num_classes=num_classes)
image_batch: torch.Tensor = torch.rand(8, 1, 28, 28)
logits: torch.Tensor = model(image_batch)

print(f"Images [N, C, H, W]: {image_batch.shape}")
print(f"Logits [N, K]: {logits.shape}")
print(f"First convolution weights: {model.conv1.weight.shape}")
print(f"Second convolution weights: {model.conv2.weight.shape}")

assert logits.shape == (8, num_classes)


Images [N, C, H, W]: torch.Size([8, 1, 28, 28])
Logits [N, K]: torch.Size([8, 10])
First convolution weights: torch.Size([32, 1, 3, 3])
Second convolution weights: torch.Size([64, 32, 3, 3])


### Learning checkpoint

The complete path is

$$
\text{pixels} \rightarrow \text{learned feature maps} \rightarrow
\text{pooled features} \rightarrow \text{flattened vectors} \rightarrow
\text{class logits}.
$$

Convolutions learn **what** local patterns matter, pooling compresses **where** strong responses occur, and the linear layer combines the resulting evidence into a prediction.
